# 07 Model Evaluation

In [ ]:
import cv2
import torch
import numpy as np

from PIL import Image
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, login
from src.training.unetpp import MBENUNetPlusPlus

from src.features.prnu import PRNU
from src.features.frequency import Frequency
from src.dataset.mben import MBENFusionModule
from src.training.unetpp import MBENUNetPlusPlus
from src.features.illumination import Illumination

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/unet-ensemble')

## 07-1 Data Preparation

### Feature Extraction

In [ ]:
def extract_prnu(rgb_image: np.ndarray) -> np.ndarray:
    prnu = PRNU()
    img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)

    wavelet, means = prnu.denoise_image()
    residual = prnu.suppress_residual(wavelet, means)

    vis = prnu.visualize(residual)

    return vis

In [ ]:
def extract_frequency(rgb_image: np.ndarray) -> np.ndarray:
    freq = Frequency()
    img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)

    f_spectrum = freq.compute_frequency_spectrum(img)

    return f_spectrum

In [ ]:
def extract_illumination(rgb_image: np.ndarray, sigma, window_size) -> np.ndarray:
    illum = Illumination()
    img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
    
    mean = illum.get_mean(img, sigma)
    var = illum.get_variance(img, window_size)
    combined = illum.blend(mean, var)

    return combined

### Preprocess

In [ ]:
def _to_gray_uint8(arr: np.ndarray) -> np.ndarray:
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr

In [ ]:
def _norm_tensor(arr: np.ndarray) -> torch.Tensor:
    arr = arr.astype(np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0)

In [ ]:
def preprocess(rgb_array: np.ndarray, img_size):
    h, w = img_size, img_size

    prnu_raw = extract_prnu(rgb_array)
    freq_raw = extract_frequency(rgb_array)
    illu_raw = extract_illumination(rgb_array)

    def _resize(arr):
        img = Image.fromarray(_to_gray_uint8(arr)).resize((w, h), Image.BILINEAR)
        return np.array(img, dtype=np.uint8)

    prnu = _resize(prnu_raw)
    freq = _resize(freq_raw)
    illu = _resize(illu_raw)

    prnu_t = _norm_tensor(prnu)
    illu_t = _norm_tensor(illu)
    freq_t = _norm_tensor(freq)

    fused_t = torch.cat([prnu_t, illu_t, freq_t], dim=0)

    return (
        prnu_t.unsqueeze(0),   # (1, 1, H, W)
        illu_t.unsqueeze(0),   # (1, 1, H, W)
        freq_t.unsqueeze(0),   # (1, 1, H, W)
        fused_t.unsqueeze(0),  # (1, 3, H, W)
    )


### Load Model

In [ ]:
def load_model(
    safetensors_path: str,
    device: torch.device,
    from_hub: bool = False,
    hub_repo: str = None,
    hub_filename: str = "model.safetensors",
    hub_token: str = None,
) -> MBENUNetPlusPlus:
    
    if from_hub:
        if not hub_repo:
            raise ValueError("hub_repo must be provided when from_hub=True.")
        print(f"Downloading model from HuggingFace: {hub_repo}/{hub_filename} ...")
        safetensors_path = hf_hub_download(
            repo_id   =hub_repo,
            filename  =hub_filename,
            token     =hub_token,
        )
        print(f"Downloaded to: {safetensors_path}")

    model = MBENUNetPlusPlus(mben_out_ch=64)
    state_dict = load_file(safetensors_path, device=str(device))
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"Model loaded from: {safetensors_path}")
    return model

## 07-2 Models

### U-Net++ Model

In [ ]:
def predict(
    model: MBENUNetPlusPlus,
    image_path: str,
    image_size: int,
    device: torch.device,
    threshold: float = 0.5,
) -> dict:
    
    rgb_array = np.array(Image.open(image_path).convert("RGB"))

    prnu_t, illu_t, freq_t, fused_t = preprocess(rgb_array, image_size)
    prnu_t  = prnu_t.to(device)
    illu_t  = illu_t.to(device)
    freq_t  = freq_t.to(device)
    fused_t = fused_t.to(device)

    with torch.no_grad():
        logits = model(prnu_t, illu_t, freq_t, fused_t)   # (1, 1, H, W)

    prob_map    = torch.sigmoid(logits).squeeze().cpu().numpy()          # (H, W)
    binary_mask = (prob_map >= threshold).astype(np.uint8) * 255        # (H, W)

    return {
        "prob_map":    prob_map,
        "binary_mask": binary_mask,
    }


## Run

In [ ]:
IMAGE_PATH    = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Normalized/Authentic/template-a/img341.jpg"
OUTPUT_PATH   = "/content/drive/Shareddrives/U-Net Ensemble/Output"
THRESHOLD     = 0.5
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH    = "/content/model.safetensors"

FROM_HUB      = True
HUB_REPO      = "your-username/mben-unetpp"
HUB_FILENAME  = "model.safetensors"
HUB_TOKEN     = "read-token"

IMAGE_SIZE = 256

In [ ]:
print(f"Using device: {DEVICE}")

model = load_model(
    safetensors_path=MODEL_PATH,
    device=DEVICE,
    from_hub=FROM_HUB,
    hub_repo=HUB_REPO,
    hub_filename=HUB_FILENAME,
    hub_token=HUB_TOKEN,
)
result = predict(model, IMAGE_PATH, IMAGE_SIZE, DEVICE, threshold=THRESHOLD)

# Save binary mask
mask_img = Image.fromarray(result["binary_mask"])
mask_img.save(OUTPUT_PATH)
print(f"Binary mask saved to: {OUTPUT_PATH}")

# Print quick stats
coverage = result["binary_mask"].mean() / 255 * 100
print(f"Mask coverage: {coverage:.2f}% of image")